# Runtime screening of the full method catalogue on ACSIncome

Companion to `07_acsincome_runtime.ipynb`. Where notebook 07 compares three representatives against the original authors' code, this notebook screens **every pre-processing method in the package** on ACSIncome subsets of 50,000 and 100,000 samples, checking that each implementation behaves and scales as expected -- the ratio between the two timings should sit around 2 for the roughly linear pipelines that dominate this catalogue (values well above that suggest a super-linear implementation worth inspecting).

Scope notes:

- `OptimizedPreproc` is excluded: it requires dataset-specific distortion constraints over discrete features, and its optimisation cost is governed by the size of the discrete value domain rather than by the number of samples.
- `ReweighingClassifier` / `FairBalanceClassifier` are excluded: they are thin meta-estimators whose pre-processing step is exactly the `Reweighing` / `FairBalance` computation screened here, plus the training of the wrapped classifier.
- `FairMask` is a meta-estimator, so its `fit` necessarily includes training its internal models (extrapolation models plus the default estimator); the timing reflects that.
- Each measurement is a single run: this is a screening pass, not a benchmark.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

from skfair.datasets import fetch_acs_income
from skfair.preprocessing import (
    FAWOS,
    DisparateImpactRemover,
    FairBalance,
    FairMask,
    FairOversampling,
    FairSmote,
    FairwayRemover,
    HeterogeneousFOS,
    LearningFairRepresentations,
    Massaging,
    Reweighing,
)

RANDOM_STATE = 42
SIZES = [50_000, 100_000]


def subset(n):
    X, y = fetch_acs_income(subsample=n, random_state=RANDOM_STATE)
    return X, np.asarray(y)


DATA = {n: subset(n) for n in SIZES}
REPAIR_COLS = [c for c in DATA[SIZES[0]][0].columns if c not in ("SEX", "RAC1P")]
print({n: X.shape for n, (X, y) in DATA.items()})

{50000: (50000, 10), 100000: (100000, 10)}


In [2]:
# name -> (constructor, call kind); constructors follow the package's
# registry defaults (sens_attr/priv_group supplied as in experiments)
METHODS = {
    "Reweighing": (lambda: Reweighing(sens_attr="SEX"), "weight"),
    "FairBalance": (lambda: FairBalance(sens_attr="SEX"), "weight"),
    "Massaging": (lambda: Massaging(sens_attr="SEX", priv_group=1), "resample"),
    "DisparateImpactRemover": (
        lambda: DisparateImpactRemover(
            sens_attr="SEX", repair_columns=REPAIR_COLS, lambda_param=1.0
        ),
        "transform",
    ),
    "FairwayRemover": (
        lambda: FairwayRemover(sens_attr="SEX", priv_group=1),
        "resample",
    ),
    "FairMask": (lambda: FairMask(sens_attr="SEX", random_state=RANDOM_STATE), "fit"),
    "FairOversampling": (
        lambda: FairOversampling(
            sens_attr="SEX", priv_group=1, random_state=RANDOM_STATE
        ),
        "resample",
    ),
    "HeterogeneousFOS": (
        lambda: HeterogeneousFOS(sens_attr="SEX", random_state=RANDOM_STATE),
        "resample",
    ),
    "FAWOS": (
        lambda: FAWOS(sens_attr="SEX", priv_group=1, random_state=RANDOM_STATE),
        "resample",
    ),
    "FairSmote": (
        lambda: FairSmote(sens_attr="SEX", random_state=RANDOM_STATE),
        "resample",
    ),
    "LearningFairRepresentations": (
        lambda: LearningFairRepresentations(
            sens_attr="SEX", priv_group=1, random_state=RANDOM_STATE
        ),
        "transform",
    ),
}


def run_once(method_factory, kind, X, y):
    m = method_factory()
    t0 = time.perf_counter()
    if kind == "resample":
        m.fit_resample(X, y)
    elif kind == "transform":
        m.fit(X, y).transform(X)
    elif kind == "weight":
        m.fit_transform(X, y)
    elif kind == "fit":
        m.fit(X, y)
    return time.perf_counter() - t0

In [3]:
rows = []
for name, (factory, kind) in METHODS.items():
    entry = {"method": name}
    for n in SIZES:
        X, y = DATA[n]
        try:
            seconds = run_once(factory, kind, X, y)
            entry[n] = seconds
            print(f"{name:28s} n={n:>8,d}  {seconds:9.2f}s", flush=True)
        except Exception as exc:  # record and continue screening
            entry[n] = np.nan
            print(f"{name:28s} n={n:>8,d}  FAILED: {exc!r}", flush=True)
    rows.append(entry)

Reweighing                   n=  50,000       0.02s


Reweighing                   n= 100,000       0.03s


FairBalance                  n=  50,000       0.02s


FairBalance                  n= 100,000       0.03s


Massaging                    n=  50,000       0.18s


...

FAWOS                        n= 100,000      31.46s


FairSmote                    n=  50,000      32.70s


FairSmote                    n= 100,000      64.50s


LearningFairRepresentations  n=  50,000     115.56s


LearningFairRepresentations  n= 100,000     310.48s


In [4]:
res = pd.DataFrame(rows).set_index("method")
res.columns = [f"{n // 1000}k (s)" for n in SIZES]
res["ratio"] = res.iloc[:, 1] / res.iloc[:, 0]
res["scaling"] = np.where(res["ratio"] > 3, "CHECK", "ok")

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
res.to_csv(out_dir / "acsincome_screening.csv")

res.round(2)

,50k (s),100k (s),ratio,scaling
method,,,,
Reweighing,0.02,0.03,1.26,ok
FairBalance,0.02,0.03,1.44,ok
Massaging,0.18,0.45,2.46,ok
DisparateImpactRemover,0.23,0.42,1.83,ok
FairwayRemover,0.14,0.27,1.99,ok
FairMask,26.99,76.23,2.82,ok
FairOversampling,19.50,39.37,2.02,ok
HeterogeneousFOS,53.21,115.45,2.17,ok
FAWOS,13.22,31.46,2.38,ok


## Reading the table

`ratio` is the 100k/50k timing quotient: ~2 for linear methods, slightly above for the neighbour-based samplers ($n \log n$). Entries marked `CHECK` scale super-linearly and deserve a look at the implementation.